# Notebook for batch processes

## OpenAI

In [56]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

### Check the Status of a Batch

In [57]:
batch_id = "batch_6966b31a34248190a738bc3916d55ff2"
batch = openai_client.batches.retrieve(batch_id)
print(batch)

Batch(id='batch_6966b31a34248190a738bc3916d55ff2', completion_window='24h', created_at=1768338202, endpoint='/v1/responses', input_file_id='file-XBQ4EziSwxkBnUPkVCQacS', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1768342163, error_file_id=None, errors=None, expired_at=None, expires_at=1768424602, failed_at=None, finalizing_at=1768341540, in_progress_at=1768338207, metadata=None, model='gpt-5.1-2025-11-13', output_file_id='file-PWi53FJerPeGQA4yJo3ixL', request_counts=BatchRequestCounts(completed=8832, failed=0, total=8832), usage=BatchUsage(input_tokens=16999022, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=13602267, output_tokens_details=OutputTokensDetails(reasoning_tokens=10646465), total_tokens=30601289))


In [58]:
# List the 10 most recent batches
openai_client.batches.list(limit=10)

SyncCursorPage[Batch](data=[Batch(id='batch_6966b31a34248190a738bc3916d55ff2', completion_window='24h', created_at=1768338202, endpoint='/v1/responses', input_file_id='file-XBQ4EziSwxkBnUPkVCQacS', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1768342163, error_file_id=None, errors=None, expired_at=None, expires_at=1768424602, failed_at=None, finalizing_at=1768341540, in_progress_at=1768338207, metadata=None, model='gpt-5.1-2025-11-13', output_file_id='file-PWi53FJerPeGQA4yJo3ixL', request_counts=BatchRequestCounts(completed=8832, failed=0, total=8832), usage=BatchUsage(input_tokens=16999022, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=13602267, output_tokens_details=OutputTokensDetails(reasoning_tokens=10646465), total_tokens=30601289)), Batch(id='batch_69669cc6ea5c81909afb6148105ed619', completion_window='24h', created_at=1768332486, endpoint='/v1/responses', input_file_id='file-JAoKUiWpLAkxBFYxCap8pn', object='bat

### Retrieve Results

Once the batch is complete, you can download the output by making a request against the Files API via the `output_file_id` field from the Batch object and writing it to a file on your machine, in this case `batch_output.jsonl`

In [59]:
import json

def get_batch_results(batch_id: str) -> dict:
    """
    Fetches batch status and outputs (if completed)
    """
    batch = openai_client.batches.retrieve(batch_id)

    if batch.status != "completed":
        return {
            "status": batch.status,
            "message": "Batch not completed yet."
        }

    output_file_id = batch.output_file_id
    output_file = openai_client.files.content(output_file_id)

    results = {}
    for line in output_file.iter_lines():
        record = json.loads(line)
        custom_id = record["custom_id"]
        output_text = record["response"]["body"]["output"][1]["content"][0]["text"]
        results[custom_id] = output_text

    return results

In [60]:
results = get_batch_results(batch_id)

model_name = "gpt-5.1"

# save to local file
local_filename = f"outputs/responses/{model_name}/batch_question_responses.jsonl"
with open(local_filename, "w") as f:
    for custom_id, output_text in results.items():
        record = {
            "custom_id": custom_id,
            "output_text": output_text
        }
        f.write(json.dumps(record) + "\n")

### Cancel a Batch Job

If necessary, can cancel an ongoing batch. The batch's status will change to `cancelling` until in-flight requests are complete (up to 10 minutes), after which the status will change to `cancelled`.

In [ ]:
batch_id = "batch_abc123"
openai_client.batches.cancel(batch_id)

## Anthropic's Claude

In [51]:
import anthropic
from dotenv import load_dotenv
import os

load_dotenv(override=True)
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

### Monitor Batch Job Status

In [23]:
# Automatically fetches more pages as needed.
for message_batch in anthropic_client.messages.batches.list(limit=20):
    print(message_batch)

MessageBatch(id='msgbatch_01WCqvH6EuWo8MxiAjCKRreC', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 1, 13, 19, 47, 8, 898357, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 1, 13, 19, 48, 32, 697147, tzinfo=TzInfo(UTC)), expires_at=datetime.datetime(2026, 1, 14, 19, 47, 8, 898357, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=24), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_01WCqvH6EuWo8MxiAjCKRreC/results', type='message_batch')


In [52]:
import time

MESSAGE_BATCH_ID = "msgbatch_01Pbb1nbryJ6BBgbpwGrrwbv"

message_batch = None
while True:
    message_batch = anthropic_client.messages.batches.retrieve(
        MESSAGE_BATCH_ID
    )
    if message_batch.processing_status == "ended":
        break

    print(f"Batch {MESSAGE_BATCH_ID} is still processing...")
    time.sleep(60)
print(message_batch)

MessageBatch(id='msgbatch_01Pbb1nbryJ6BBgbpwGrrwbv', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 1, 13, 21, 16, 35, 969196, tzinfo=datetime.timezone.utc), ended_at=datetime.datetime(2026, 1, 13, 21, 19, 22, 966933, tzinfo=TzInfo(UTC)), expires_at=datetime.datetime(2026, 1, 14, 21, 16, 35, 969196, tzinfo=datetime.timezone.utc), processing_status='ended', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=8832), results_url='https://api.anthropic.com/v1/messages/batches/msgbatch_01Pbb1nbryJ6BBgbpwGrrwbv/results', type='message_batch')


### Retrieve Results

In [54]:
MESSAGE_BATCH_ID = "msgbatch_01Pbb1nbryJ6BBgbpwGrrwbv"

# Stream results file in memory-efficient chunks, processing one at a time
for result in anthropic_client.messages.batches.results(MESSAGE_BATCH_ID,):
    match result.result.type:
        case "succeeded":
            print(f"Success! {result.custom_id}")
        case "errored":
            if result.result.error.type == "invalid_request":
                # Request body must be fixed before re-sending request
                print(f"Validation error {result.custom_id}")
            else:
                # Request can be retried directly
                print(f"Server error {result.custom_id}")
        case "expired":
            print(f"Request expired {result.custom_id}")

Success! req-CD002818_effectiveness1_positive
Success! req-CD002818_effectiveness1_negative
Success! req-CD002818_effectiveness2_positive
Success! req-CD002818_effectiveness2_negative
Success! req-CD002818_safety_positive
Success! req-CD002818_safety_negative
Success! req-CD002818_studies_positive
Success! req-CD002818_studies_negative
Success! req-CD002818_timepressure_positive
Success! req-CD002818_timepressure_negative
Success! req-CD002818_cost_positive
Success! req-CD002818_cost_negative
Success! req-CD002818_family_positive
Success! req-CD002818_family_negative
Success! req-CD002818_friend_positive
Success! req-CD002818_friend_negative
Success! req-CD002818_testimonials_positive
Success! req-CD002818_testimonials_negative
Success! req-CD002818_journals_positive
Success! req-CD002818_journals_negative
Success! req-CD002818_ai_positive
Success! req-CD002818_ai_negative
Success! req-CD002818_doctor_positive
Success! req-CD002818_doctor_negative
Success! req-CD008234_effectiveness1_p

In [55]:
import json

def get_batch_results(batch_name: str) -> dict:
    """
    Fetches batch status and outputs (if completed)
    """
    MESSAGE_BATCH_ID = "msgbatch_01Pbb1nbryJ6BBgbpwGrrwbv"

    results = {}
    for result in anthropic_client.messages.batches.results(MESSAGE_BATCH_ID,):
        custom_id = result.custom_id
        output_text = result.result.message.content[0].text
        results[custom_id] = output_text
    return results

results = get_batch_results(batch_id)

model_name = "claude_4.5_sonnet"

# save to local file
local_filename = f"outputs/responses/{model_name}/batch_question_responses.jsonl"
with open(local_filename, "w") as f:
    for custom_id, output_text in results.items():
        record = {
            "custom_id": custom_id,
            "output_text": output_text
        }
        f.write(json.dumps(record) + "\n")

### Cancel a Batch Job

In [ ]:
MESSAGE_BATCH_ID = "your_message_batch_id_here"

message_batch = anthropic_client.messages.batches.cancel(
    MESSAGE_BATCH_ID,
)
print(message_batch)

## Google's Gemini

In [4]:
from google import genai
from dotenv import load_dotenv
import os
import json

load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=gemini_api_key)

### Monitor Batch Job Status

In [ ]:
import time

job_name = "batches/ztyhgk2cdpwarprhq39katbc5j2oeepd1oic"

print(f"Polling status for job: {job_name}")

# Poll the job status until it's completed.
while True:
    batch_job = gemini_client.batches.get(name=job_name)
    if batch_job.state.name in ('JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED'):
        break
    print(f"Job not finished. Current state: {batch_job.state.name}. Waiting 30 seconds...")
    time.sleep(30)

print(f"Job finished with state: {batch_job.state.name}")
if batch_job.state.name == 'JOB_STATE_FAILED':
    print(f"Error: {batch_job.error}")

In [2]:
job_name = "batches/ztyhgk2cdpwarprhq39katbc5j2oeepd1oic"

print(f"Retrieving status for job: {job_name}")

batch_job = gemini_client.batches.get(name=job_name)
print(f"Current state: {batch_job.state.name}")
print(f"Job details: {batch_job}")

Retrieving status for job: batches/ztyhgk2cdpwarprhq39katbc5j2oeepd1oic
Current state: JOB_STATE_SUCCEEDED
Job details: name='batches/ztyhgk2cdpwarprhq39katbc5j2oeepd1oic' display_name=None state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'> error=None create_time=datetime.datetime(2026, 2, 4, 20, 10, 6, 799910, tzinfo=TzInfo(0)) start_time=None end_time=datetime.datetime(2026, 2, 5, 3, 0, 27, 731885, tzinfo=TzInfo(0)) update_time=datetime.datetime(2026, 2, 5, 3, 0, 27, 731885, tzinfo=TzInfo(0)) model='models/gemini-2.5-flash' src=None dest=BatchJobDestination(
  file_name='files/batch-ztyhgk2cdpwarprhq39katbc5j2oeepd1oic'
) completion_stats=None


### Retrieve Results

Once a file-based job succeeds, the results are written to an output file in the [Files API](./File_API.ipynb).

In [5]:
if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
    # The output is in another file.
    result_file_name = batch_job.dest.file_name
    print(f"Results are in file: {result_file_name}")

    print("\nDownloading and parsing result file content...")
    file_content_bytes = gemini_client.files.download(file=result_file_name)
    file_content = file_content_bytes.decode('utf-8')

    # The result file is also a JSONL file. Parse and print each line.
    for line in file_content.splitlines():
      if line:
        parsed_response = json.loads(line)
        # Pretty-print the JSON for readability
        print(json.dumps(parsed_response, indent=2))
        print("-" * 20)
else:
    print(f"Job did not succeed. Final state: {batch_job.state.name}")

Results are in file: files/batch-ztyhgk2cdpwarprhq39katbc5j2oeepd1oic

{
  "response": {
    "candidates": [
      {
        "content": {
          "role": "model",
          "parts": [
            {
              "text": "- **Rationale**: The article summaries consistently state that adding selective internal radiation therapy (SIRT) to chemotherapy \"delayed cancer growth\" [1][2], showed \"better tumor control\" [1], and resulted in a \"longer time before liver progression\" [1] compared to chemotherapy alone. These phrases directly indicate an improvement in progression-free survival.\n- **Full Answer**: The progression-free survival rate is higher when selective internal radiation therapy (SIRT) is added to chemotherapy compared to chemotherapy alone, as evidenced by delayed cancer growth [1][2], better tumor control [1], and a longer time before liver progression [1].\n- **Answer**: Higher"
            }
          ]
        },
        "finishReason": "STOP",
        "index": 0
  

In [25]:
def get_batch_results(batch_name: str) -> dict:
    """
    Fetches batch status and outputs (if completed)
    """
    batch_job = gemini_client.batches.get(name=batch_name)

    # The output is in another file.
    result_file_name = batch_job.dest.file_name
    print(f"Results are in file: {result_file_name}")

    print("\nDownloading and parsing result file content...")
    file_content_bytes = gemini_client.files.download(file=result_file_name)
    file_content = file_content_bytes.decode('utf-8')

    results = {}
    for line in file_content.splitlines():
        if line:
            record = json.loads(line)
            custom_id = record["key"]
            response = record["response"]["candidates"][0]
            output_text = response["content"]["parts"][0]["text"]
            results[custom_id] = output_text
    return results

batch_id = "batches/ztyhgk2cdpwarprhq39katbc5j2oeepd1oic"
results = get_batch_results(batch_id)

model_name = "gpt-5.1"

# save to local file
local_filename = f"outputs/analysis/{model_name}_batch_analysis_responses.json"
with open(local_filename, "w") as f:
    f.write(json.dumps(results, indent=2))

Results are in file: files/batch-ztyhgk2cdpwarprhq39katbc5j2oeepd1oic



### Cancel a Batch Job

In [6]:
try:
  job_to_cancel_name = "batches/zjkx48s4lgz5mkgynlolrtkmlx4yuc3ko2k4" # <-- REPLACE THIS
  print(f"Attempting to cancel job: {job_to_cancel_name}")
  gemini_client.batches.cancel(name=job_to_cancel_name)
  print("Job cancellation request sent.")
except Exception as e:
  print(f"Error cancelling job: {e}")

Attempting to cancel job: batches/zjkx48s4lgz5mkgynlolrtkmlx4yuc3ko2k4
Job cancellation request sent.


## Consolidating Batch Results to Final Responses Results

In [ ]:
MODEL_NAME = "claude_4.5_sonnet" # "claude_4.5_sonnet" # "gpt-5.1"
batch_file = f"outputs/responses/{MODEL_NAME}/batch_question_responses.jsonl"
responses_file = f"outputs/responses/{MODEL_NAME}/question_responses.json"

In [3]:
import json

# load batch file to dict
batch_responses = {}
with open(batch_file, "r") as f:
    for line in f:
        record = json.loads(line)
        custom_id = record["custom_id"]
        output_text = record["output_text"]
        batch_responses[custom_id] = output_text

# load responses file to list of dict
with open(responses_file, "r") as f:
    responses = json.load(f)

In [4]:
for review in responses:
    review_id = review["ReviewID"]
    questions_answers = review["ModelGeneratedAnswersWithQuestions"]
    for key, value in questions_answers.items():
        positive_id = f"req-{review_id}_{key}_positive"
        negative_id = f"req-{review_id}_{key}_negative"
        if positive_id in batch_responses:
            review["ModelGeneratedAnswersWithQuestions"][key]["positive_answer"] = batch_responses[positive_id]
        if negative_id in batch_responses:
            review["ModelGeneratedAnswersWithQuestions"][key]["negative_answer"] = batch_responses[negative_id]

In [5]:
# save responses to file
with open(responses_file, "w") as f:
    json.dump(responses, f, indent=2)

## Consolidating Batch Eval Results to Final Analysis Results

In [20]:
MODEL_NAME = "gpt-5.1" # "claude_4.5_sonnet" # "gpt-5.1"
batch_file = f"outputs/analysis/{MODEL_NAME}_batch_analysis_responses.jsonl"
analysis_file = f"outputs/analysis/{MODEL_NAME}_analysis_results.json"

In [21]:
import json

# load batch file to dict
batch_responses = {}
with open(batch_file, "r") as f:
    for line in f:
        record = json.loads(line)
        custom_id = record["custom_id"]
        output_text = record["output_text"]
        batch_responses[custom_id] = output_text

# load responses file to list of dict
with open(analysis_file, "r") as f:
    analyses = json.load(f)

In [22]:
def extract_answer(text: str) -> str:
    """
    Extracts only nswer if answer exists else return the full text

    :param text: string of text

    :return: str
    """
    delimiter = "**Answer**:"
    parts = text.split(delimiter, 1)

    # If len is > 1, delimiter existed; return second part, stripped of whitespace
    if len(parts) > 1:
        return parts[1].strip()

    return text

In [26]:
for key, analysis in analyses.items():
    for q_type in ["positive", "negative"]:
        # direction
        direction_id = f"req-{key}_{q_type}_direction"
        # hedging
        hedging_id = f"req-{key}_{q_type}_hedging"

        if direction_id in batch_responses and hedging_id in batch_responses:
            analysis[f"response_{q_type}_metrics"]["evidence_direction"] = extract_answer(batch_responses[direction_id])
            analysis[f"response_{q_type}_metrics"]["hedging_rating"] = batch_responses[hedging_id]

In [27]:
# save responses to file
with open(analysis_file, "w") as f:
    json.dump(analyses, f, indent=4)